# Selective Transmon Rx Calibration

## Prerequisites

This guide assumes you have configured a `DeviceSetup` and created `BosonicQubit` objects with assigned parameters. Please see [our tutorials](https://docs.zhinst.com/labone_q_user_manual/applications_library/tutorials/index.html) if you need to create your setup and qubits for the first time.

You can run this notebook on real hardware in the lab. If you don't have the hardware at your disposal, you can also run the notebook "as is" using an emulated session (see below).

Before running the selective transmon Rx calibration experiment, the following calibrations must have been performed:

- **Transmon spectroscopy**: the steady-state ancilla transmon resonance frequency `transmon_resonance_frequency_at_n0` must be known.
- **Dispersive shift**: the dispersive shift `chi` between the memory cavity and the ancilla transmon must be measured, so that each number-selective drive line is tuned to the correct frequency.
- **Displacement calibration**: the conversion factor `displacement_amp_per_unit_beta` must be known (see the [Displacement Calibration how-to guide](02_displacement_calibration.ipynb)), to prepare coherent states for photon numbers $n > 0$.

If you are just getting started with the LabOne Q Applications Library, please don't hesitate to reach out to us at info@zhinst.com.

## Background

In this how-to guide, you will calibrate the **number-selective transmon $\pi$ and $\pi/2$ pulse amplitudes** for a bosonic qubit using the `selective_transmon_rx_calibration` experiment workflow in the LabOne Q Applications Library.

### Number-selective transmon drives

Due to the dispersive interaction between the ancilla transmon and the memory cavity, the transmon transition frequency depends on the photon number $n$ in the cavity:

$$f_\mathrm{transmon}(n) = f_\mathrm{transmon}(0) + n \cdot \chi$$

This photon-number-dependent frequency splitting allows a selective $\pi$ pulse — a drive resonant at $f_\mathrm{transmon}(n)$ — to flip the transmon only when the cavity contains exactly $n$ photons, leaving all other Fock states unaffected.

Each photon number $n$ requires independently calibrated pulse amplitudes, because the selective drive lines operate at different frequencies (`drive_transmon_at_n0`, `drive_transmon_at_n1`, …).

### Pulse sequence

For each photon number $n$, the experiment prepares the memory cavity in a coherent state with mean photon number $\bar{n} = n$ using the displacement operation, then sweeps the selective drive amplitude and measures the transmon:

$$\text{qubit} \;---\; [D(\bar{n}=n)] \;---\; [R_x(\mathrm{amp},\, n)] \;---\; [\mathrm{measure}]$$

For $n = 0$, no displacement is applied — the cavity remains in vacuum. For $n > 0$, the displacement prepares a coherent state $|\beta\rangle$ with $|\beta|^2 = n$. The Rabi oscillation contrast at photon number $n$ is proportional to the Poisson population $P(n|\beta)$ of the prepared state; this reduces the signal contrast for higher $n$ but does not affect the oscillation period, so the extracted $\pi$ amplitude is still accurate.

All photon numbers are swept in a single experiment run using a nested sweep: the outer sweep steps through the photon numbers and the inner sweep steps through the drive amplitudes. Both sweeps run in real time.

### Calibration outputs

The workflow produces two calibration constants per qubit and per photon number $n$:

| Parameter | Stored in | Description |
|-----------|-----------|-------------|
| `amp_pi` | `qubit.parameters.selective_transmon_X180_amplitudes[n]` | Selective $\pi$-pulse amplitude |
| `amp_pi2` | `qubit.parameters.selective_transmon_X90_amplitudes[n]` | Selective $\pi/2$-pulse amplitude |

## Imports

You'll start by importing `laboneq.simple`, as well as the `BosonicQubit` and `BosonicQubitOperations` classes from the bosonic qubits contribution.

In [ ]:
import copy

import numpy as np
from laboneq.simple import *

from laboneq_applications.contrib.qpu_types.bosonic_qubits import (
    BosonicQubit,
    BosonicQubitOperations,
)

## Define your experimental setup

Let's define our experimental setup. We will need:

- a [DeviceSetup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html)
- one or more [BosonicQubits](https://docs.zhinst.com/labone_q_user_manual/applications_library/reference/contrib/index.html)
- a set of `BosonicQubitOperations`
- a [QPU](https://docs.zhinst.com/labone_q_user_manual/core/reference/dsl/quantum.html#laboneq.dsl.quantum.qpu.QPU)

Here we will be brief and mainly provide the code to obtain these objects. For full details, see the tutorials on [defining your experimental setup](../../../tutorials/sources/getting_started.ipynb) and [logbooks and data saving](../../../tutorials/sources/logbooks.ipynb).

We will use **2 bosonic qubits** in this guide. Change this number to match your setup.

### Instrument serial numbers

In [ ]:
# Replace with the serial number of your SHFQC instrument
SHFQC_SERIAL = "DEV12073"

DATASERVER_HOST = "localhost"
DATASERVER_PORT = "8004"

number_of_qubits = 2

### DeviceSetup

The selective transmon Rx calibration experiment requires signal lines for the memory cavity drive, the number-selective transmon drives, and qubit readout. Here we use a single **SHFQC** instrument.

Each bosonic qubit uses:
- One **QA channel** pair (OUTPUT + INPUT) for readout
- One **SG channel** for the memory cavity drive (`drive_memory`) — used to prepare coherent Fock states via displacement for photon numbers $n > 0$
- One **SG channel** for all number-selective transmon drives (`drive_transmon_at_n0` … `drive_transmon_at_n{max_photon_number}`)
- Optionally, one **SG channel** in LF mode for the SWAP drive (`swap_drive`) — not used in this experiment, but included so the same `DeviceSetup` can be reused across all bosonic qubit experiments

If you already have a `DeviceSetup`, you can reuse it and skip this cell. Just make sure the signal line names match those expected by `BosonicQubit`.

In [ ]:
# Maximum photon number — determines how many selective transmon signal lines are created.
MAX_PHOTON_NUMBER = 5

device_setup = DeviceSetup(uid="my_bosonic_setup")
device_setup.add_dataserver(
    uid="my_bosonic_setup",
    host=DATASERVER_HOST,
    port=DATASERVER_PORT,
)

device_setup.add_instruments(
    SHFQC(
        uid="device_shfqc",
        address=SHFQC_SERIAL,
        device_options="SHFQC/LRT/PLUS/QC6CH/RTR",
    ),
)

# ── Signal lines for qubit q0 ─────────────────────────────────────────
device_setup.add_connections(
    "device_shfqc",
    # Readout (QA channel 0)
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT"),
    # Memory cavity drive (SG channel 0) — displacement for Fock state preparation
    create_connection(to_signal="q0/drive_memory", ports="SGCHANNELS/0/OUTPUT"),
    # SWAP drive — not used here, included for setup reuse
    create_connection(to_signal="q0/swap_drive", ports="SGCHANNELS/4/OUTPUT"),
)
# Number-selective transmon drive lines for q0 (all share SG channel 2)
for n in range(MAX_PHOTON_NUMBER + 1):
    device_setup.add_connections(
        "device_shfqc",
        create_connection(
            to_signal=f"q0/drive_transmon_at_n{n}",
            ports="SGCHANNELS/2/OUTPUT",
        ),
    )

# ── Signal lines for qubit q1 (optional) ─────────────────────────────
if number_of_qubits == 2:
    device_setup.add_connections(
        "device_shfqc",
        create_connection(to_signal="q1/measure", ports="QACHANNELS/0/OUTPUT"),
        create_connection(to_signal="q1/acquire", ports="QACHANNELS/0/INPUT"),
        create_connection(to_signal="q1/drive_memory", ports="SGCHANNELS/1/OUTPUT"),
        create_connection(to_signal="q1/swap_drive", ports="SGCHANNELS/5/OUTPUT"),
    )
    for n in range(MAX_PHOTON_NUMBER + 1):
        device_setup.add_connections(
            "device_shfqc",
            create_connection(
                to_signal=f"q1/drive_transmon_at_n{n}",
                ports="SGCHANNELS/3/OUTPUT",
            ),
        )

### BosonicQubits

We will create `BosonicQubit` objects from the logical signal groups in our `DeviceSetup` and assign qubit parameters. The qubit UID will match the name of the logical signal group (`q0`, `q1`, …).

Adjust the parameter values to reflect the properties of your device. The parameters relevant to selective transmon Rx calibration are highlighted in the comments.

In [ ]:
# Qubit parameters — adjust these to match your device.
# Parameters marked (*) are directly relevant to the selective Rx calibration experiment.
QUBIT_PARAMS = {
    # ── Memory cavity ────────────────────────────────────────────────
    "memory_lo_frequency": 5.8e9,
    "memory_resonance_frequency": 6.0e9,
    "memory_drive_amplitude": 0.5,
    "memory_drive_length": 0.1e-6,  # (*) Displacement pulse duration
    "memory_drive_pulse": {"function": "gaussian", "sigma": 0.25},
    "memory_spectroscopy_amplitude": 0.5,
    "memory_spectroscopy_length": 4e-6,
    "memory_drive_range": 10,
    # ── Ancilla transmon ─────────────────────────────────────────────
    "transmon_lo_frequency": 4.8e9,
    "transmon_resonance_frequency_at_n0": 5.0e9,  # (*) From transmon spectroscopy
    "transmon_drive_range": 10,
    # ── Selective transmon Rx drives (*) ─────────────────────────────
    # These are the parameters this experiment calibrates.
    # The initial values are starting guesses; the experiment will update them.
    "max_photon_number": MAX_PHOTON_NUMBER,
    "selective_transmon_Rx_length": 0.08e-6,  # (*) Selective pulse duration
    "selective_transmon_Rx_pulse": {"function": "gaussian", "sigma": 0.25},
    "selective_transmon_X180_amplitudes": [0.8]
    * (MAX_PHOTON_NUMBER + 1),  # (*) To be calibrated
    "selective_transmon_X90_amplitudes": [0.4]
    * (MAX_PHOTON_NUMBER + 1),  # (*) To be calibrated
    # ── SWAP drive ────────────────────────────────────────────────────
    # This setup drives the SWAP line from an SG channel, which is put into
    # LF mode with its local oscillator at 0 Hz. Leave this out for a SWAP
    # line on a baseband channel, such as an HDAWG output.
    "swap_drive_on_sg_channel": True,
    # ── SWAP gate ─────────────────────────────────────────────────────
    "swap_length": 500e-9,
    "swap_amplitude": 0.1,
    "swap_pulse": {"function": "const"},
    "swap_drive_range": 5,
    # ── Readout ───────────────────────────────────────────────────────
    "readout_lo_frequency": 6.8e9,
    "readout_resonator_frequency": 7.0e9,
    "readout_amplitude": 0.5,
    "readout_length": 1e-6,
    "readout_pulse": {"function": "const"},
    "readout_integration_length": 1e-6,
    "readout_integration_delay": 20e-9,
    # ── Reset ─────────────────────────────────────────────────────────
    "reset_delay_length": 1e-6,
    # ── Memory-transmon coupling (*) ──────────────────────────────────
    "chi": -1e6,  # (*) Dispersive shift in Hz
    # ── Displacement calibration (*) ─────────────────────────────────
    "displacement_amp_per_unit_beta": 0.2,  # (*) Must be calibrated before this experiment
    # ── SNAP gate parameters (populated by a later calibration) ───────
    "snap_drive_amp_per_n": {},
    "snap_drive_phase_per_n": {},
    "snap_drive_phase_slope_sign_per_n": {},
}

In [ ]:
q0 = BosonicQubit.from_logical_signal_group(
    uid="q0",
    lsg=device_setup.logical_signal_groups["q0"],
    parameters=copy.deepcopy(QUBIT_PARAMS),
)
qubits = [q0]

if number_of_qubits == 2:
    q1 = BosonicQubit.from_logical_signal_group(
        uid="q1",
        lsg=device_setup.logical_signal_groups["q1"],
        parameters=copy.deepcopy(QUBIT_PARAMS),
    )
    qubits.append(q1)

for q in qubits:
    print("-------------")
    print("Qubit UID:", q.uid)
    print("Qubit logical signals:")
    for sig, lsg in q.signals.items():
        print(f"  {sig:<30} ('{lsg}')")

### Quantum Operations

Create the set of `BosonicQubitOperations`:

In [ ]:
qops = BosonicQubitOperations()

### QPU

Create the `QPU` object from the qubits and the quantum operations:

In [ ]:
from laboneq.dsl.quantum import QPU

qpu = QPU(qubits, quantum_operations=qops)

## Connect to Session

In [ ]:
session = Session(device_setup)
session.connect(do_emulation=True)  # set do_emulation=False when at a real setup

## Create a `FolderStore` for Saving Data

The experiment `Workflows` can automatically save the inputs and outputs of all their tasks to the folder path we specify when instantiating the `FolderStore`. Here, we choose the current working directory.

In [ ]:
from pathlib import Path

folder_store = workflow.logbook.FolderStore(Path.cwd() / "data")

We disable saving in this guide. To enable it, simply run `folder_store.activate()`.

In [ ]:
folder_store.deactivate()

## Optional: Configure the LoggingStore

You can also activate/deactivate the `LoggingStore`, which controls the display of `Workflow` logging information in the notebook. See the [tutorial on Recording Experiment Workflow Results](../../../tutorials/sources/logbooks.ipynb) for details.

Displaying the `Workflow` logging information is activated by default, but here we deactivate it to shorten the outputs, which are not very meaningful in emulation mode.

**We recommend that you do not deactivate the Workflow logging in practice.**

In [ ]:
from laboneq.workflow.logbook import LoggingStore

logging_store = LoggingStore()
logging_store.deactivate()

## Running the Experiment Workflow

You'll now instantiate the experiment workflow and run it. For more details on what experiment workflows are and what tasks they execute, see the [Experiment Workflows tutorial](../../../tutorials/sources/experiment_workflows.ipynb).

Start by importing the `selective_transmon_rx_calibration` experiment workflow from the bosonic qubits contribution, as well as `plot_simulation` for inspecting the pulse sequence.

In [ ]:
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation

from laboneq_applications.contrib.experiments.bosonic_qubits import (
    selective_transmon_rx_calibration,
)

### Configure the experiment options

Let's create the options object for the selective transmon Rx calibration workflow and inspect it using `workflow.show_fields`:

In [ ]:
options = selective_transmon_rx_calibration.experiment_workflow.options()
workflow.show_fields(options)

Notice that, unless we change it:

- The experiment is run in `AcquisitionType.INTEGRATION` and `AveragingMode.CYCLIC`, using 1024 averages (`count`).
- The analysis workflow runs automatically after the experiment (`do_analysis=True`). It fits a cosine model to the Rabi oscillation for each (qubit, photon number) pair and returns the calibrated `amp_pi` and `amp_pi2`.
- The figures produced by the analysis are automatically closed (`close_figures=True`)
- The qubit parameters will not be updated (`update=False`).

The `create_experiment` sub-options expose additional pulse overrides for advanced use:
- `selective_drive_pulse` / `selective_drive_length`: shape and duration overrides for the selective transmon drive pulse.
- `memory_drive_pulse` / `memory_drive_length`: shape and duration overrides for the displacement pulse.

Here, let's reduce the number of averages for a quicker run in emulation mode, and disable closing the figures produced by the analysis so we see them in the cell output. Note that the fits attempted by the analysis in emulation mode are not representative, because we do not acquire data from a real experiment.

In [ ]:
options.count(1)
options.close_figures(False)

### Define the photon numbers and amplitude sweep

Unlike the SNAP calibration (which operates on one photon number at a time), this experiment calibrates all requested photon numbers in a single run. The `photon_numbers_memory` argument accepts a list — for example `[0, 1, 2, 3]` — and the hardware sweeps them in a single compiled sequence using a real-time outer sweep.

Define a single amplitude sweep that is shared across all photon numbers. The sweep should span the expected range of $\pi$-pulse amplitudes. A good starting range is $[0,\; 2 \times A_{\pi,\mathrm{estimate}}]$, where $A_{\pi,\mathrm{estimate}}$ is the current best guess (or a rough design estimate).

In [ ]:
# Photon numbers to calibrate in a single run
PHOTON_NUMBERS = [0, 1, 2, 3]

qubit = qpu.quantum_elements[0]

# Amplitude sweep — same array used for all photon numbers.
# Span [0, 2 x estimate] so the pi amplitude sits in the middle of the sweep,
# capped at the maximum pulse amplitude of 1.
amp_estimate = qubit.parameters.selective_transmon_X180_amplitudes[0]
amplitudes_q0 = np.linspace(0, min(2 * amp_estimate, 1.0), 21)

print(f"Photon numbers : {PHOTON_NUMBERS}")
print(
    f"Amplitude sweep: {amplitudes_q0[0]:.3f} to {amplitudes_q0[-1]:.3f} "
    f"({len(amplitudes_q0)} points)"
)
print(
    f"Total sweep points per qubit: "
    f"{len(PHOTON_NUMBERS)} photon numbers x {len(amplitudes_q0)} amplitudes "
    f"= {len(PHOTON_NUMBERS) * len(amplitudes_q0)} points"
)

In [ ]:
exp_workflow = selective_transmon_rx_calibration.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=[qubit.uid],
    amplitudes_transmon=[amplitudes_q0],
    photon_numbers_memory=PHOTON_NUMBERS,
    options=options,
)

workflow_results = exp_workflow.run()

### Inspect the Tasks That Were Run

The selective transmon Rx calibration workflow executes four steps in sequence: `create_experiment`, `compile_experiment`, `run_experiment`, and (when `do_analysis=True`) the `analysis_workflow`, which runs the `fit_data`, `extract_qubit_parameters` and plotting tasks. With `update=True`, an `update_qpu` task then stores the new parameter values in the `qpu`.

In [ ]:
for t in workflow_results.tasks:
    print(t)

### Inspect the Output Simulation

You can inspect the compiled experiment and plot the simulated output to verify that the pulse sequence looks as expected. The selective Rabi sequence consists of a displacement pulse on the `drive_memory` line (for $n > 0$), a selective $R_x$ pulse on the `drive_transmon_at_n{n}` line, and a readout pulse.

The experiment uses a nested sweep: the outer real-time sweep steps through the photon numbers (`dsl.match` / `dsl.case`), and the inner real-time sweep steps through the amplitudes. All `dsl.match` branches share the same real-time window (padded to the longest branch including the displacement pulse), so the sequence has the same duration for all photon numbers.

The total duration of one real-time repetition is approximately:

$$T \approx t_\mathrm{displacement} + t_{Rx} + t_\mathrm{readout} + t_\mathrm{reset}$$

In [ ]:
q = qpu.quantum_elements[0]
total_time = (
    q.parameters.memory_drive_length
    + q.parameters.selective_transmon_Rx_length
    + q.parameters.readout_length
    + q.parameters.reset_delay_length
)

compiled_experiment = workflow_results.tasks["compile_experiment"].output
plot_simulation(
    compiled_experiment,
    length=total_time * 1.05,
)

### Inspect the Source Code of the Pulse-Sequence Creation Task

You can inspect the source code of the `create_experiment` task to see exactly how the pulse sequence is constructed using quantum operations. To learn more about quantum operations, see the [Quantum Operations tutorial](../../../tutorials/sources/quantum_operations.ipynb).

In [ ]:
selective_transmon_rx_calibration.create_experiment.src

## Analyse the Results

Because `do_analysis=True` by default, the analysis workflow ran automatically after the experiment. For each (qubit, photon number) pair it performed the following steps:

1. Retrieved the transmon signal from the result handle `{qubit_uid}/rabi_n{n}`.
2. Estimated the Rabi oscillation frequency from an FFT of the detrended signal.
3. Fitted the cosine model
$$s(x) = \mathrm{offset} + A \cdot \cos(2\pi f x + \varphi)$$
using the FFT estimate as the initial frequency guess.
4. Extracted the $\pi$ and $\pi/2$ amplitudes:
$$A_\pi = \frac{1}{2|f|}, \qquad A_{\pi/2} = \frac{1}{4|f|}$$

Two plots are generated automatically for each qubit:
- **Selective Rabi oscillations** — a grid of Rabi traces per photon number, with cosine fits and dashed lines marking $A_\pi$ and $A_{\pi/2}$.
- **Amplitude summary** — $A_\pi$ and $A_{\pi/2}$ versus photon number, with annotated values.

### Access the Analysis Outputs

In [ ]:
# Access the fit results of the analysis workflow.
analysis_result = workflow_results.tasks["analysis_workflow"].tasks["fit_data"].output

q_uid = qpu.quantum_elements[0].uid
q_result = analysis_result[q_uid]

# Keys available in the per-qubit result dict:
print("Available result keys:", list(q_result.keys()))

The per-qubit result dictionary contains:

| Key | Description |
|-----|-------------|
| `"amplitudes"` | 1D array of swept selective drive amplitudes |
| `"photon_numbers"` | List of photon numbers that were calibrated |
| `"rabi_data"` | Dict keyed by `str(n)` — per-photon-number Rabi oscillation data and fit results |
| `"amp_pi_vs_n"` | 1D array of fitted $\pi$ amplitudes, one entry per photon number (same order as `photon_numbers`) |
| `"amp_pi2_vs_n"` | 1D array of fitted $\pi/2$ amplitudes, one entry per photon number |

Each entry of `rabi_data[str(n)]` contains:

| Key | Description |
|-----|-------------|
| `"raw_data"` | 1D Rabi oscillation signal |
| `"fit_params"` | Fitted `[amplitude, frequency, phase, offset]` (empty if the fit failed) |
| `"fit_success"` | `True` if the fit converged |
| `"amp_pi"` | Calibrated $\pi$ amplitude (`NaN` if fit failed) |
| `"amp_pi2"` | Calibrated $\pi/2$ amplitude (`NaN` if fit failed) |
| `"fit_x"`, `"fit_y"` | Dense fit curve arrays for custom plotting |

### Extract the Calibrated Amplitudes

In [ ]:
amp_pi_vs_n = q_result["amp_pi_vs_n"]
amp_pi2_vs_n = q_result["amp_pi2_vs_n"]

print(f"Qubit {q_uid}:")
print(f"{'n':>4}  {'amp_pi':>10}  {'amp_pi2':>10}  {'fit_ok':>8}")
print("-" * 40)
for i, n in enumerate(PHOTON_NUMBERS):
    fit_ok = q_result["rabi_data"][str(n)]["fit_success"]
    print(f"  {n:>2}  {amp_pi_vs_n[i]:>10.4f}  {amp_pi2_vs_n[i]:>10.4f}  {fit_ok!s:>8}")

### Update the Qubit Parameters

The analysis workflow returns the extracted qubit parameters together with their previous values. Set `options.update(True)` to store the new values in the `qpu` automatically when the workflow runs. Here, we update the `qpu` manually instead. Qubits without a valid fit have no new values and keep their previous parameters.

The new values are the qubit's `selective_transmon_X180_amplitudes` and `selective_transmon_X90_amplitudes` lists, with the entries for the successfully fitted photon numbers replaced. These parameters are indexed by photon number, so `selective_transmon_X180_amplitudes[n]` holds the $\pi$ amplitude when the cavity contains $n$ photons.

Once updated, the `rx` quantum operation (and the operations built on it, such as `x180` and `x90`) will use these amplitudes by default.

In [ ]:
from pprint import pprint

qubit_parameters = workflow_results.tasks["analysis_workflow"].output
pprint(qubit_parameters)

qpu.update(qubit_parameters["new_parameter_values"])

## Parallel Selective Transmon Rx Calibration

The `selective_transmon_rx_calibration` workflow natively supports running multiple qubits in parallel. When you pass a list of qubit UIDs together with corresponding lists of amplitude arrays, the pulse sequences for all qubits are compiled into a single experiment and executed simultaneously on the hardware.

Each qubit gets its own independently chosen amplitude sweep, so qubits with different expected $\pi$ amplitudes can be characterised in one shot. The only constraint is that all amplitude arrays must have the **same number of points**, because the hardware sweeps them in lock-step. Both qubits must also share the same `photon_numbers_memory` list.

Here we demonstrate a parallel calibration of photon numbers `[0, 1, 2, 3]` on both qubits:

In [ ]:
# Store the calibrated values in the qpu automatically.
options.update(True)

q0 = qpu.quantum_elements[0]
q1 = qpu.quantum_elements[1]

PHOTON_NUMBERS = [0, 1, 2, 3]

# Each qubit gets its own sweep, spanning [0, 2 x its amplitude estimate] (capped at 1).
amp_estimate_q0 = q0.parameters.selective_transmon_X180_amplitudes[0]
amp_estimate_q1 = q1.parameters.selective_transmon_X180_amplitudes[0]

amplitudes_q0 = np.linspace(0, min(2 * amp_estimate_q0, 1.0), 21)
amplitudes_q1 = np.linspace(0, min(2 * amp_estimate_q1, 1.0), 21)

exp_workflow_2q = selective_transmon_rx_calibration.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=[q0.uid, q1.uid],
    amplitudes_transmon=[amplitudes_q0, amplitudes_q1],
    photon_numbers_memory=PHOTON_NUMBERS,
    options=options,
)

workflow_results_2q = exp_workflow_2q.run()

### Check the Updated Qubit Parameters

With `update=True`, the workflow stored the new parameter values in the `qpu`. Qubits without a valid fit keep their previous parameters:

In [ ]:
qubit_parameters_2q = workflow_results_2q.tasks["analysis_workflow"].output
pprint(qubit_parameters_2q["new_parameter_values"])

for q in [q0, q1]:
    print(
        f"{q.uid}: selective_transmon_X180_amplitudes = {q.parameters.selective_transmon_X180_amplitudes}"
    )
    print(
        f"{q.uid}: selective_transmon_X90_amplitudes = {q.parameters.selective_transmon_X90_amplitudes}"
    )

Great! You have now calibrated the number-selective transmon $\pi$ and $\pi/2$ pulse amplitudes for all qubits. The `selective_transmon_X180_amplitudes` and `selective_transmon_X90_amplitudes` parameters are stored in the qubit objects and will be used by all subsequent experiments that require number-selective transmon control — including the SNAP gate calibration and Wigner tomography.

The next step in the bosonic qubit tune-up sequence is the [**SWAP gate calibration**](04_swap_calibration.ipynb), which calibrates the transmon-memory SWAP used to prepare Fock states.